# GPT From Scratch

A consolidated build of the GPT model from first principles, synthesizing chapters 1–5 of Raschka's *Build a Large Language Model From Scratch*.

Covers: tokenization → embeddings → attention → transformer block → full GPT model → pretraining loop

In [1]:
import sys
print(sys.executable)   # full path to the python binary the kernel is using
print(sys.version)      # python version string

/Users/tanvidey3/Desktop/dev/WebDevLearning/.venv/bin/python
3.12.13 (main, Jun 23 2026, 15:44:24) [Clang 22.1.3 ]


In [2]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [4]:
%pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.0/983.0 kB 25.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn
import tiktoken

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

tokenizer = tiktoken.get_encoding("gpt2")

PyTorch version: 2.13.0
CUDA available: False
Using device: cpu


# Table of contents
1. Config          — GPT_CONFIG_124M dict, hyperparams
2. Training Loop   — train_model_simple (references things not yet defined)
3. Loss Functions  — calc_loss_batch, calc_loss_loader, evaluate_model
4. Data Pipeline   — GPTDatasetV1, create_dataloader_v1
5. GPT Model       — GPTModel
6. Transformer Block — TransformerBlock
7. Attention       — CausalAttention, MultiHeadAttention
8. Building Blocks — LayerNorm, GELU, FeedForward
9. Generation      — generate, text_to_token_ids, token_ids_to_text
10. Run            — one cell that wires config + data + model + training


### SELF NOTES
writing order will be written like this - so others can understand in which order I wrote this

```python
# w.o:01: - created the GPT config dic 
```

# 1 - Config

In [6]:

# w.o:01: - created the GPT config dic
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers" : 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}


In [7]:
# w.o:01 - a -  more helper functions
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dim - # (n_tokens,) -> (batch, n_tokens)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # drop the batch dim
    return tokenizer.decode(flat.tolist())


# Create Data Loader

In [8]:
# w.o:02: download the main txt data
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [9]:
# w.o:05: - GPTDataset class to create the input ids and target ids
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i: i+max_length]
            target_chunk = token_ids[i+1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [10]:
# w.o:04: create dataloader function
def create_dataloader(txt, tokenizer, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    # create dataset
    dataset = GPTDataset(txt, tokenizer, max_length, stride)

    # create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers
    )

    return dataloader


In [11]:
# w.o:03: create the data loader for train and va;
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)

train_loader = create_dataloader(
    train_data,
    tokenizer,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader(
    val_data,
    tokenizer,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

# Device

In [12]:
# w.o:11: creating the device var that we will use to have the model and other items in one device example cpu or gpu
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")


print(f"Using {device} device.")

Using mps device.


# GPT Model code

## Layer norm

In [13]:
# w.o:13: Layer Norm
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x-mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift
        


## Feed Forward Network

In [14]:
# w.o:17: GELU activation
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [15]:
# w.o:16: feed forward class
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

## Multi-head attention

In [16]:
# w.o:15: multi head attention class
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0 , "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_in, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        # x shape : (batch, num_tokens, emb_dim)
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)    # shape : (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # we split the matrix by adding a `num_heads` dimension
        # we unroll the last dim : (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # compute scaled dot-product attention (self-attention) with a causal mask
        # transpose is done for (num_tokens, head_dim) -> (head_dim, num_tokens)
        attn_scores = queries @ keys.transpose(2,3) # dot product for each head

        # original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # use the mask to fill attention scores
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # shape (b, num_tokens, num_heads, head_dim) after transpose the dim changed
        context_vec = (attn_weights @ values).transpose(1,2)

        # combine heads where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

## Transformer Block code

In [17]:
# w.o:14: main transformer code
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads = cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x) # shape : (batch, num_tokens, emb_dim)
        x = self.drop_shortcut(x)
        x = x + shortcut    # add the original input back

        # shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

## GPT Model class

In [18]:
# w.o:12: main GPT model code starts here
# In comments using example batch=2, seq_len=4, emb_dim=768, vocab_size=50257:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # create the embedding of token and position
        # example shape : (batch, seq_length, emb_dim) -> (2, 4, 768) for tok_emb and pos_emb
        # This are the main weights
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])


        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"])

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape # shape : (2,4) (batch, seq_length)
        
        # each integer -> row lookup in (vocab_size, emb_dim) table
        # (2, 4) -> (2, 4, 768)
        tok_embeds = self.tok_emb(in_idx)
        
        # pos_emb(arrange(4)): [0,1,2,3] -> row lookup in (1024, 768) table -> (4, 768) no batch axis
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        # (2, 4, 768) + (4, 768) : (2, 4, 768) pos_embeds broadcasts across batch axis
        x = tok_embeds + pos_embeds

        # 10% zeroed during training, no-op at inference - no shape change
        x = self.drop_emb(x)

        # trf blocks (12 x TransformerBlock):
        # Each block: norm -> attentoon -> residual -> norm -> FFN -> residual
        # Each block is shape preserving (2, 4, 768) -> (2,4, 768)
        # applied 12 times in sequence
        x = self.trf_blocks(x)

        # final norm - Layer norm over the last dim, per token - no shape change
        # (2, 4, 768)
        x = self.final_norm(x)

        # Linear (768 -> 50257) i.e. (2,4,768) x (768, 50257) = (2, 4, 50257)
        # now each token will have pred score for each of the vocab
        # each row a token and each value is the prob for that vocab
        logits = self.out_head(x)

        return logits
        

# Loss Functions

In [19]:
# w.o:08: run a single forward pass and return the cross entropy loss as a scalar tensor (with autograd graph still attached)
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch) # doing the inference with model here
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0,1),    # collapsing the dim of 0,1 since this was mainly first used for image
        target_batch.flatten(),
    )
    return loss


In [20]:
# w.o:07 - a - average cross-entropy loss across
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(len(data_loader), num_batches)
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i<num_batches:
            # calculate the loss for each batch
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    # at the end we are just giving the average after calculating loss for each batch we are summing it up and dividing by the total number of batches
    return total_loss / num_batches

# Training loop

In [21]:
# w.o:08 - a - compute the train and val losses across eval_iter batches
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
        model.train()
        return train_loss, val_loss       

In [22]:
# w.o:10: -  generate token ids
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # crop current context if it exceeds the supported context size
        # e.g,. if LLM supports only 5 tokens and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # focus only on the last time step
        # Logitshspae - (batch, n_token, vocab_size) becomes (batch, vocab_size)
        # this says take 2nd dim i.e. n_token last one and have all the vocab size pred for all the batch
        logits = logits[:, -1, :]

        # get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)   # now the shape is (batch, 1)

        # append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1) # (batch, n_tokens+1)

    return idx


In [23]:
# w.o:09: - generate 50 tokens of text starting from `start_context` and print the result on a single line
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]    # this will help us not to hardcode this variable
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size
        )
        decoded_text = token_ids_to_text(token_ids, tokenizer)
        print(decoded_text.replace("\n", " "))
        # we need to move back to training mode
        model.train()

In [24]:
# w.o:07: - plain training loop with periodic evaluation and per epoch sample generation
def train_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
                
        # when each epoch ends
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen

# Run the training - wiring everything up

In [ ]:
# w.o:06: - wrote the main code cell that runs the training and wires everything up
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_loss, val_loss, tokens_seen = train_model(
    model, 
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs= num_epochs,
    eval_freq=5,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer

)

Ep 1 (Step 000000): Train loss 9.874, Val loss 10.007
Ep 1 (Step 000005): Train loss 8.032, Val loss 8.294
Every effort moves you the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
Ep 2 (Step 000010): Train loss 6.711, Val loss 7.040
Ep 2 (Step 000015): Train loss 5.964, Val loss 6.597
Every effort moves you the the to the"" a""" a the a the. """""-- the""""". """""" the""" to the""""""""
Ep 3 (Step 000020): Train loss 5.793, Val loss 6.548
Ep 3 (Step 000025): Train loss 5.622, Val loss 6.512
Every effort moves you know to have to the                                             
Ep 4 (Step 000030): Train loss 5.321, Val loss 6.469
Ep 4 (Step 000035): Train loss 5.627, Val loss 6.629
Every effort moves you                                                  
Ep 5 (Step 000040): Train loss 5.481, Val loss 6.627
Every effort moves you           

# saving model

In [51]:
# w.o:18:  knowing the model state dict and saving it
sd = model.state_dict()
print(list(sd.keys())[:4])
print(sd["tok_emb.weight"].shape)
print(sd["trf_blocks.0.att.W_query.weight"].shape)

['tok_emb.weight', 'pos_emb.weight', 'trf_blocks.0.att.mask', 'trf_blocks.0.att.W_query.weight']
torch.Size([50257, 768])
torch.Size([768, 768])


In [52]:
torch.save(model.state_dict(), "model.pth")

# Loading the model

In [25]:
# w.o:19: loading the model
model = GPTModel(GPT_CONFIG_124M)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval()

Device: mps


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

# Saving model optimizer state

In [ ]:
# w.o:20: saving the model optimizer
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
    },
    "model_and_optimizer.pth"
)

# Advanced checkpointing (rotating checkpoints + resume support)

## Checkpoint saving - Helper function

In [29]:
# w.o:21: checkpoint saving
import os
import glob

# folder where all checkpoints get written
CHECKPOINT_DIR = "checkpoints"
# save a checkpoint every N epochs (1 = every epoch, 5 = every 5th epoch, 0 = never save periodically)
SAVE_EVERY_N_EPOCHS = 1
KEEP_LAST_N_CHECKPOINTS = 3

def save_checkpoint(model, optimizer, epoch, global_step, train_losses, val_losses, track_tokens_seen, checkpoint_dir, keep_last_n):
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"checkpoint_epoch{epoch:03d}_step{global_step:06d}.pt"
    )

    torch.save({
        "epoch":epoch,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict":optimizer.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "track_tokens_seen": track_tokens_seen
    }, checkpoint_path)

    print(f"Saved checkpoint: {checkpoint_path}")

    existing_checkpoints = sorted(glob.glob(os.path.join(checkpoint_dir, "checkpoint_epoch*.pt")))
    if keep_last_n > 0 and len(existing_checkpoints) > keep_last_n:
        for old_checkpoint in existing_checkpoints[:-keep_last_n]:
            os.remove(old_checkpoint)
            print(f"Removed old checkpoint (exceeded keep_last_n={keep_last_n}): {old_checkpoint}")

    return checkpoint_path


## Find latest checkpoint

In [26]:
# w.o:22: find latest checkpoint
def find_latest_checkpoint(checkpoint_dir):
    if not os.path.exists(checkpoint_dir):
        return None
    
    existing_checkpoints = sorted(glob.glob(os.path.join(checkpoint_dir, "checkpoint_epoch*.pt")))
    if not existing_checkpoints:
        return None

    latest = existing_checkpoints[-1]
    print(f"Found latest checkpoint: {latest}")
    return latest

In [27]:
# w.o:23: 
def load_checkpoint(checkpoint_path, model, optimizer, device):
    """
    Load a checkpoint into model + optimizer in place, and return where training left off.
    """
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1   # resume on the epoch AFTER the one that was saved
    global_step = checkpoint["global_step"]
    train_losses = checkpoint["train_losses"]
    val_losses = checkpoint["val_losses"]
    track_tokens_seen = checkpoint["track_tokens_seen"]

    print(f"Resumed from checkpoint: {checkpoint_path}")
    print(f"  Continuing at epoch {start_epoch}, global_step {global_step}")

    return start_epoch, global_step, train_losses, val_losses, track_tokens_seen


In [30]:
# w.o:22: training loop with automatic checkpoint saving and resume support
def train_model_with_checkpoints(model, train_loader, val_loader, optimizer, device,
                                  num_epochs, eval_freq, eval_iter, start_context, tokenizer,
                                  checkpoint_dir=CHECKPOINT_DIR,
                                  save_every_n_epochs=SAVE_EVERY_N_EPOCHS,
                                  keep_last_n_checkpoints=KEEP_LAST_N_CHECKPOINTS,
                                  resume=True):

    start_epoch = 0
    global_step = -1
    train_losses, val_losses, track_tokens_seen = [], [], []

    if resume:
        latest_checkpoint = find_latest_checkpoint(checkpoint_dir)
        if latest_checkpoint is not None:
            start_epoch, global_step, train_losses, val_losses, track_tokens_seen = load_checkpoint(
                latest_checkpoint, model, optimizer, device
            )
        else:
            print("No existing checkpoint found, starting training from scratch.")

    tokens_seen = track_tokens_seen[-1] if track_tokens_seen else 0

    for epoch in range(start_epoch, num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

        if save_every_n_epochs > 0 and (epoch + 1) % save_every_n_epochs == 0:
            save_checkpoint(model, optimizer, epoch, global_step, train_losses, val_losses,
                             track_tokens_seen, checkpoint_dir, keep_last_n_checkpoints)

    # always save a final checkpoint at the end, regardless of save_every_n_epochs
    save_checkpoint(model, optimizer, num_epochs - 1, global_step, train_losses, val_losses,
                     track_tokens_seen, checkpoint_dir, keep_last_n_checkpoints)

    return train_losses, val_losses, track_tokens_seen



In [67]:
# w.o:23: run training with checkpointing, resuming automatically if a checkpoint already exists
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

train_loss, val_loss, tokens_seen = train_model_with_checkpoints(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=20,
    eval_freq=5,
    eval_iter=5,
    start_context="Every effort moves you",
    tokenizer=tokenizer,
    checkpoint_dir="checkpoints",
    save_every_n_epochs=1,
    keep_last_n_checkpoints=3,
    resume=True
)

Found latest checkpoint: checkpoints/checkpoint_epoch009_step000089.pt
Resumed from checkpoint: checkpoints/checkpoint_epoch009_step000089.pt
  Continuing at epoch 10, global_step 89
Ep 11 (Step 000090): Train loss 0.173, Val loss 6.852
Ep 11 (Step 000095): Train loss 0.149, Val loss 6.890
Every effort moves you?"  "Yes--quite insensible to the irony. She wanted him vindicated--and by me!"  He laughed again, and threw back the window-curtains, moved aside a _jardiniere_ full of
Saved checkpoint: checkpoints/checkpoint_epoch010_step000098.pt
Removed old checkpoint (exceeded keep_last_n=3): checkpoints/checkpoint_epoch007_step000071.pt
Ep 12 (Step 000100): Train loss 0.131, Val loss 6.935
Ep 12 (Step 000105): Train loss 0.133, Val loss 6.978
Every effort moves you?"  "Yes--quite insensible to the irony. She wanted him vindicated--and by me!"  He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I
Saved checkpoint: checkpoints/checkpoint_

# Inference the model - Top-K Sampling & Temperature

In [36]:
# w.o:24: generate() — top-k + temperature + eos_id support
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    """
    Generate tokens autoregressively.
    Args:
        model         : trained GPTModel
        idx           : (1, T) tensor of starting token ids
        max_new_tokens: how many new tokens to generate
        context_size  : model's max context window (crops idx if too long)
        temperature   : 0.0 = greedy (argmax), >0.0 = sample from softmax distribution
                        lower  → more deterministic / repetitive
                        higher → more random / creative
        top_k         : if set, only the top_k highest-logit tokens are kept before sampling
                        helps avoid low-probability garbage tokens
        eos_id        : if the model produces this token id, stop early
                        e.g. tokenizer.encode("<|endoftext|>")[0]
    """
    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature

            # New (not in book): numerical stability tip to get equivalent results on mps device
            # subtract rowwise max before softmax
            logits = logits - logits.max(dim=-1, keepdim=True).values

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

        

In [37]:
# w.o:25a: load the latest checkpoint for inference — reusing find_latest_checkpoint()
model_inf = GPTModel(GPT_CONFIG_124M)
model_inf.to(device)

latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)

if latest_ckpt is not None:
    checkpoint = torch.load(latest_ckpt, map_location=device, weights_only=True)
    model_inf.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded from checkpoint: {latest_ckpt}")
    print(f"  epoch={checkpoint['epoch']}, global_step={checkpoint['global_step']}")
else:
    # fallback to model.pth if no checkpoint exists yet
    model_inf.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    print("No checkpoint found — loaded from model.pth")

model_inf.eval()
print("Ready for inference.")


Found latest checkpoint: checkpoints/checkpoint_epoch019_step000179.pt
Loaded from checkpoint: checkpoints/checkpoint_epoch019_step000179.pt
  epoch=19, global_step=179
Ready for inference.


In [38]:
# w.o:25b: thin wrapper so we don't repeat boilerplate in every demo cell
def sample(prompt, max_new_tokens=50, temperature=0.0, top_k=None):
    idx = text_to_token_ids(prompt, tokenizer).to(device)
    token_ids = generate(
        model=model_inf,
        idx=idx,
        max_new_tokens=max_new_tokens,
        context_size=GPT_CONFIG_124M["context_length"],
        temperature=temperature,
        top_k=top_k,
        eos_id=tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
    )
    return token_ids_to_text(token_ids, tokenizer)


In [39]:
# w.o:26: compare greedy vs different temperature + top_k combos
torch.manual_seed(42)

PROMPT = "Every effort moves you"

configs = [
    {"label": "greedy (temp=0.0)",          "temperature": 0.0,  "top_k": None},
    {"label": "temp=0.7, top_k=50",         "temperature": 0.7,  "top_k": 50},
    {"label": "temp=1.0, top_k=50",         "temperature": 1.0,  "top_k": 50},
    {"label": "temp=1.5, top_k=100",        "temperature": 1.5,  "top_k": 100},
]

print(f'Prompt: "{PROMPT}"\n{"-"*70}')
for cfg in configs:
    out = sample(PROMPT, max_new_tokens=50, temperature=cfg["temperature"], top_k=cfg["top_k"])
    print(f'\n[{cfg["label"]}]')
    print(out)


Prompt: "Every effort moves you"
----------------------------------------------------------------------

[greedy (temp=0.0)]
Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I

[temp=0.7, top_k=50]
Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I

[temp=1.0, top_k=50]
Every effort moves you villain'd never touched a brush."

And his tone told me in a flash that he never thought of anything else.

I moved away, instinctively embarrassed by my unexpected discovery; and as I turned, my eye fell on a small picture

[temp=1.5, top_k=100]
Every effort moves you had been to go a little years of his pictures-- irony. He called up all Gisburn's past! Usuallyia's face watching the fr